In [60]:
import torch 
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

In [61]:
with open('input.txt','r',encoding='utf-8') as f: # with statement replaces a try-catch block with a concise shorthand
    # ensures closing resources right after processing them.
    text=f.read()
print(f"length of the dataset characters {len(text)}")

length of the dataset characters 1115393


In [62]:
# pick out the unique characters
chars=sorted(list(set(text)))
vocab_size=len(chars)
print("".join(chars))


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [63]:
# creating mapping from char to integers
stoi={ch:i for i,ch in enumerate(chars)} # enumerate gives index(not key) value
itos={i:ch for i,ch in enumerate(chars)}
# for index_int,val in enumerate (stoi):
#     print(f"key: {val} value: {stoi[val]} ")


# lambda takes lambda var: -----> var as input to the function 
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers 
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [64]:
import torch 
data= torch.tensor(encode(text), dtype=torch.long)
print(data.shape)
print(data[:10])
# print(decode(data[0][:10].item()))


torch.Size([1115393])
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])


In [65]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

def tensor_decoder(ten):
    dec=decode([ten[x].item() for x in range(ten.shape[0])])
    return dec
    

# dec=[data[x].item() for x in range(10)]
# decode(dec)

In [66]:
block_size = 8 # also called the context size
train_data[:block_size+1] # +1 because if lets say abcd is the len is 3, d context is abc 


# context can be taken anywhere between 1 to 8 is the meaning

x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [67]:
# print(len(data))
batch_size=4
# print(torch.randint(len(data) - block_size, (batch_size,)))
ix = torch.randint(len(data) - block_size, (batch_size,))
print([data[i:i+block_size] for i in ix])

[tensor([56, 43, 40, 43, 50,  0, 13, 52]), tensor([39, 45, 43,  8,  0,  0, 31, 43]), tensor([42,  6,  1, 42, 43, 41, 50, 47]), tensor([50,  1, 58, 46, 56, 53, 52, 43])]


In [68]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) # catches starting point randomly from any line
    x = torch.stack([data[i:i+block_size] for i in ix])   ## they become a row in 4x8 tensor
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs(xb):')
print(xb.shape)
print(xb)
print('targets(yb):')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")
        
        
### Done

inputs(xb):
torch.Size([4, 8])
tensor([[53, 59,  6,  1, 58, 56, 47, 40],
        [49, 43, 43, 54,  1, 47, 58,  1],
        [13, 52, 45, 43, 50, 53,  8,  0],
        [ 1, 39,  1, 46, 53, 59, 57, 43]])
targets(yb):
torch.Size([4, 8])
tensor([[59,  6,  1, 58, 56, 47, 40, 59],
        [43, 43, 54,  1, 47, 58,  1, 58],
        [52, 45, 43, 50, 53,  8,  0, 26],
        [39,  1, 46, 53, 59, 57, 43,  0]])
----
when input is [53] the target: 59
when input is [53, 59] the target: 6
when input is [53, 59, 6] the target: 1
when input is [53, 59, 6, 1] the target: 58
when input is [53, 59, 6, 1, 58] the target: 56
when input is [53, 59, 6, 1, 58, 56] the target: 47
when input is [53, 59, 6, 1, 58, 56, 47] the target: 40
when input is [53, 59, 6, 1, 58, 56, 47, 40] the target: 59
when input is [49] the target: 43
when input is [49, 43] the target: 43
when input is [49, 43, 43] the target: 54
when input is [49, 43, 43, 54] the target: 1
when input is [49, 43, 43, 54, 1] the target: 47
when input is [

In [69]:
# x = torch.randn(1, 3)
# x.shape
# torch.stack((x,x)).shape
# >>> torch.stack((x, x)) # same as torch.stack((x, x), dim=0)
# tensor([[[ 0.3367,  0.1288,  0.2345],
#          [ 0.2303, -1.1229, -0.1863]],

#         [[ 0.3367,  0.1288,  0.2345],
#          [ 0.2303, -1.1229, -0.1863]]])
# >>> torch.stack((x, x)).size()
# torch.Size([2, 2, 3])
# >>> torch.stack((x, x), dim=1)
# tensor([[[ 0.3367,  0.1288,  0.2345],
#          [ 0.3367,  0.1288,  0.2345]],

#         [[ 0.2303, -1.1229, -0.1863],
#          [ 0.2303, -1.1229, -0.1863]]])
# >>> torch.stack((x, x), dim=2)
# tensor([[[ 0.3367,  0.3367],
#          [ 0.1288,  0.1288],
#          [ 0.2345,  0.2345]],

#         [[ 0.2303,  0.2303],
#          [-1.1229, -1.1229],
#          [-0.1863, -0.1863]]])
# >>> torch.stack((x, x), dim=-1)
# tensor([[[ 0.3367,  0.3367],
#          [ 0.1288,  0.1288],
#          [ 0.2345,  0.2345]],

#         [[ 0.2303,  0.2303],
#          [-1.1229, -1.1229],
#          [-0.1863, -0.1863]]])

In [70]:
class BigramLanguageModel(nn.Module):
    def __init__(self,vocam_size):
        super().__init__() # inherit everything from nn class
    
        self.token_embedding_table=nn.Embedding(vocab_size,vocab_size) # token embeding table of size vocab_size*vocab_size
#     def show_embedding(self):
#         print(self.token_embedding_table)
    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C) every idx pluck correspoinding row from embedding table
        # how is this giving the output as logits--> what does logits look like
        # How will the embedding table look like # B batch=4, T time is 8 and C channel is vocab size(65 in this case)
        # Pluck out the rows, arrange them in B, T, C
        # logits are the score for the next in the sequence 
        # Still confused about what is channel

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape #
#             print("from inside the class")
#             print(f"B {B}\nT: {T}\nC: {C}\n")
            logits = logits.view(B*T, C) # see the documentation --> most likely flatten portion
            targets = targets.view(B*T) # alternatively can do -1 
            loss = F.cross_entropy(logits, targets) #--> how is this working and what about C 

        return logits, loss

    def generate(self, idx, max_new_tokens): # take in context and generate +1+2 upto max tokens
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx




In [71]:
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb) ## how do we know it is going to forward 
# --> why is it going to forward by default 
print(logits.shape) # logits is 32x64 because (B*TxC)
print(loss)
# b,t,c=logits.shape
# print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.9456, grad_fn=<NllLossBackward0>)


In [72]:
m = BigramLanguageModel(vocab_size)
print(m.token_embedding_table)
print(xb.shape)

Embedding(65, 65)
torch.Size([4, 8])


In [73]:
optimizer=torch.optim.AdamW(m.parameters(),lr=1e-3)

In [74]:
batch_size=32 # this is used in get_batch--> output should be 32*8
for steps in range(1000):
    xb,yb= get_batch('train')
    
    logits, loss=m(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward() # meaning
    optimizer.step() # meaning
    
print(loss.item())

3.668074131011963


# the mathematical trick 

In [75]:
## Evertyhing from the above code is almost clear now--> atleast the tensor size part.

torch.manual_seed(1337)
B,T,C = 4,8,2
x=torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [76]:
# token talking is to summarize the interaction-- very lossy communication
# 

# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float() # between 0 to 10 with size 3x2
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [77]:
a = torch.tril(torch.ones(3, 3))
torch.sum(a, 1, keepdim=True)

tensor([[1.],
        [2.],
        [3.]])

In [78]:
# each token contain a querry and a key vector--> querry vector is what am i looking for, key vector is what do i contain
# my querry dot product with all the keys--> that dot product becomes wei(which was earlier just the average)

In [79]:
import torch
# Create a tensor
tensor = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print(tensor.shape)
# Calculate the mean of all elements in the tensor
mean_all = torch.mean(tensor)
print(f"Mean of all elements: {mean_all}")
# Calculate the mean along a specific dimension (rows)
mean_dim0 = torch.mean(tensor, dim=0)
print(f"Mean along rows (dim=0): {mean_dim0}")
# Calculate the mean along a specific dimension (columns)
mean_dim1 = torch.mean(tensor, dim=1)
print(f"Mean along columns (dim=1): {mean_dim1}")

torch.Size([2, 2])
Mean of all elements: 2.5
Mean along rows (dim=0): tensor([2., 3.])
Mean along columns (dim=1): tensor([1.5000, 3.5000])


In [80]:
torch.manual_seed(1337)
B,T,C = 4,8,2
x=torch.randn(B,T,C)
x.shape
print(x[0,1])

tensor([-0.3596, -0.9152])


In [81]:
# we want x[b,t]= mean_{i<=t} x[b,i]
xbow=torch.zeros((B,T,C)) # bow is back of words  -> word stored at each location and we are averaging

for b in range(B):
    for t in range(T):
        xprev=x[b,:t+1] # (t,:c)
        xbow[b,t] = torch.mean(xprev,0)
# print(f"xprev: {xprev}")
# print(f"xbow: {xbow}")

# OKAY
        

In [82]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [83]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

In [84]:
(0.1808-0.3596)/2

-0.0894

In [85]:
wei=torch.tril(torch.ones(T,T))
print(wei)
wei=wei/wei.sum(1,keepdim=True)
print(wei)
xbow2=wei@x # (T,T) @ (B,T,C) ---> (B(created),T,T) @ (B,T,C) --> batch multiplication 
print(xbow.shape[0])
for xin in range(xbow.shape[0]):
    print(torch.allclose(xbow[xin],xbow2[xin]))#  xbow2 is same as xbow1 

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
4
True
False
True
True


In [86]:
# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2)

False

In [87]:
# print(xbow2[1])
# print(xbow[1])
# print(torch.allclose(xbow[1],xbow2[1]))

In [88]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

False

In [89]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)


# every single token will emit querry(what i am looking for) and a key(waht do i contain)
# if key and querry are aligned then we will interact more with that token

# let's see a single Head perform self-attention
head_size = 16 # hyperparameter 
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False) # we aggregate value not tokens

# creating key and queery
k = key(x)   # (B, T, 16) 
q = query(x) # (B, T, 16)
# wei or the affinity
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)  # transpose last two dimensions
 # bash matrix multiplication

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf')) # in the decoder block this will always be present, 
                                                # in the encoder block this will be asbset
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

torch.Size([4, 8, 16])

In [90]:
########## head and all ###############

In [105]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [106]:
n_embd=32
class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__() # inherit everything from nn class
    
        self.token_embedding_table=nn.Embedding(vocab_size,n_embd) # token embeding table of size vocab_size*vocab_size
        #going from token embedding to logits we need feedforward layer
        self.position_embedding_table=nn.Embedding(block_size, n_embd) # what is the role of positional embedding
        self.lm_head = nn.Linear(n_embd, vocab_size) # lm= short for language model head
        
        
        # we are not only encoding identity of the tokens but also the position of the token
        
    #     def show_embedding(self):
    #         print(self.token_embedding_table)
    def forward(self, idx, targets=None):
        B,T=idx.shape
        
        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C) this C is embd C
        pos_embd=self.position_embedding(table(torch.arange(T))) # (T,C)--
        x= tok_emb+pos_emb # (B,T,C)---> right aliged, new dimension get added
        # x now hold not only token identity but also the position at which that token occurs
        logits= self.lm_head(tok_emb) # (B,T,C) this C is vocab size 
        
        logits = self.token_embedding_table(idx) # (B,T,C) every idx pluck correspoinding row from embedding table
        # how is this giving the output as logits--> what does logits look like
        # How will the embedding table look like # B batch=4, T time is 8 and C channel is vocab size(65 in this case)
        # Pluck out the rows, arrange them in B, T, C
        # logits are the score for the next in the sequence 
        # Still confused about what is channel

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape #
#             print("from inside the class")
#             print(f"B {B}\nT: {T}\nC: {C}\n")
            logits = logits.view(B*T, C) # see the documentation --> most likely flatten portion
            targets = targets.view(B*T) # alternatively can do -1 
            loss = F.cross_entropy(logits, targets) #--> how is this working and what about C 

        return logits, loss

    def generate(self, idx, max_new_tokens): # take in context and generate +1+2 upto max tokens
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


Notes:

- Attention is a \boldface{communication mechanism}. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with tril, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides wei by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [107]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [108]:
print(k.var())
print(q.var())
print(wei.var())

tensor(1.0931)
tensor(1.0465)
tensor(0.8911)


In [109]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

In [110]:
class Head(nn.Module):
    """ one head of self-attention """
    
  
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

#         self.dropout = nn.Dropout(dropout)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
#         logits = self.token_embedding_table(idx) # (B,T,C) every idx pluck correspoinding row from embedding table
        # how is this giving the output as logits--> what does logits look like
        # How will the embedding table look like # B batch=4, T time is 8 and C channel is vocab size(65 in this case)
        # Pluck out the rows, arrange them in B, T, C
        # logits are the score for the next in the sequence 
        # Still confused about what is channel

        # creating key and queery
        
        B,T,C=x.shape
        k = key(x)   # (B, T, 16) 
        q = query(x) # (B, T, 16)
        # wei or the affinity
        wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)  # transpose last two dimensions
         # bash matrix multiplication

        tril = torch.tril(torch.ones(T, T))
        #wei = torch.zeros((T,T))
        wei = wei.masked_fill(tril == 0, float('-inf')) # in the decoder block this will always be present, 
                                                        # in the encoder block this will be asbset
        wei = F.softmax(wei, dim=-1)

        v = value(x)
        out = wei @ v
        
        return out

    


In [111]:
# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm # sa head in video
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:] # psoitional embedding has to be upto block size
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [112]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [114]:
    
    

    
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0    
    



device='cpu'
model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    
    
    
    
    
    
    
    

    
    
    

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


0.209729 M parameters


RuntimeError: The size of tensor a (32) must match the size of tensor b (8) at non-singleton dimension 1